### Dataset ingestion and Domain normalization :

Loading a stratified 20,000 domains classifying into benign and DGA domains and isolated the Second level domain to prevent top level domain suffixes such as .com, .org, etc from manipulating character entropy and derived an output of sample domains of 5000 into sample_domains.csv

*Dataset sourced from [Prof. Christian Morbidoni's DGA_domains_dataset](https://github.com/chrmor/DGA_domains_dataset)*

In [9]:
!pip install -q shap lime xgboost scikit-learn pandas numpy matplotlib

import os
import time
import math
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)

os.makedirs("figures", exist_ok=True)
os.makedirs("data", exist_ok=True)

def extract_sld(domain: str) -> str:
    clean = str(domain).lower().strip()
    parts = clean.split(".")
    if len(parts) >= 3 and parts[-2] in {"co", "com", "org", "net", "gov", "ac", "edu"}:
        return parts[-3]
    return parts[0]

DATASET_URL = "https://raw.githubusercontent.com/chrmor/DGA_domains_dataset/master/dga_domains_full.csv"
raw_df = pd.read_csv(DATASET_URL, header=None, names=["class_str", "family", "domain"]).dropna(subset=["domain"])
raw_df["label"] = (raw_df["class_str"].str.lower() == "dga").astype(int)

benign_df = raw_df[raw_df["label"] == 0].sample(n=10000, random_state=42)
dga_df = raw_df[raw_df["label"] == 1].sample(n=10000, random_state=42)

df = pd.concat([benign_df, dga_df], ignore_index=True)
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
df["sld"] = df["domain"].apply(extract_sld)
df = df[df["sld"].str.len() > 1].reset_index(drop=True)

df.head(5000)[["domain", "sld", "family", "label"]].to_csv("data/sample_domains.csv", index=False)

print(f"Total domains: {len(df):,}")
print(df["label"].value_counts().rename({0: "Benign (0)", 1: "DGA (1)"}))
print(df[["domain", "sld", "family", "label"]].head(8).to_string(index=False))

Total domains: 20,000
label
DGA (1)       10000
Benign (0)    10000
Name: count, dtype: int64
                        domain                        sld  family  label
         vqb1rrqev5hoxavfhs.cn         vqb1rrqev5hoxavfhs  rovnix      1
                      cuny.edu                       cuny   alexa      0
          spurscommunity.co.uk             spurscommunity   alexa      0
                  albelady.com                   albelady   alexa      0
          xdjmsdlbxwqbetwgf.tw          xdjmsdlbxwqbetwgf ranbyus      1
            pbraoqwiktcwxd.com             pbraoqwiktcwxd  ramnit      1
spiritreactcommitrulereact.com spiritreactcommitrulereact  matsnu      1
                  xhasters.com                   xhasters   alexa      0


### Lexical Feature extraction :

Extracting 6 character-level lexical metrics mathematically from each SLD to capture and train our engine the patterns to distinguish benign domains from DGA domains

In [10]:
VOWELS = set("aeiou")
CONSONANT_STREAK_RE = re.compile(r"[bcdfghjklmnpqrstvwxyz]+")

def shannon_entropy(s: str) -> float:
    n = len(s)
    if n <= 1:
        return 0.0
    counts = {}
    for ch in s:
        counts[ch] = counts.get(ch, 0) + 1
    return -sum((c / n) * math.log2(c / n) for c in counts.values())

def normalized_entropy(s: str) -> float:
    n = len(s)
    if n <= 1:
        return 0.0
    return shannon_entropy(s) / math.log2(n)

def vowel_ratio(s: str) -> float:
    if not s:
        return 0.0
    return sum(1 for ch in s if ch in VOWELS) / len(s)

def digit_ratio(s: str) -> float:
    if not s:
        return 0.0
    return sum(1 for ch in s if ch.isdigit()) / len(s)

def max_consonant_streak(s: str) -> int:
    streaks = CONSONANT_STREAK_RE.findall(s)
    return max((len(st) for st in streaks), default=0)

t0 = time.perf_counter()
df["sld_length"] = df["sld"].apply(len)
df["shannon_entropy"] = df["sld"].apply(shannon_entropy)
df["norm_entropy"] = df["sld"].apply(normalized_entropy)
df["vowel_ratio"] = df["sld"].apply(vowel_ratio)
df["digit_ratio"] = df["sld"].apply(digit_ratio)
df["max_consonant_streak"] = df["sld"].apply(max_consonant_streak)
lexical_time_ms = (time.perf_counter() - t0) * 1000

LEXICAL_COLS = [
    "sld_length",
    "shannon_entropy",
    "norm_entropy",
    "vowel_ratio",
    "digit_ratio",
    "max_consonant_streak"
]

print(f"Feature extraction completed in {lexical_time_ms:.2f} ms")
print(df.groupby("label")[LEXICAL_COLS].mean().round(4).to_string())

Feature extraction completed in 260.52 ms
       sld_length  shannon_entropy  norm_entropy  vowel_ratio  digit_ratio  max_consonant_streak
label                                                                                           
0         10.6491           2.8416        0.8724       0.3609       0.0207                2.4937
1         14.1494           3.1632        0.8587       0.2749       0.0366                4.7963


### Markov Trigram model and Hybrid feature engineering matrix creation

Learns the markov character trigram from each benign training domains and computing a log probability score for each one.This is added as a seventh lexical feature along with the six earlier features and TF-IDF n-grams to produce the final training and testing inputs.

In [11]:
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

train_idx, test_idx = train_test_split(
    df.index, test_size=0.20, random_state=42, stratify=df["label"]
)

df_train = df.loc[train_idx].reset_index(drop=True)
df_test = df.loc[test_idx].reset_index(drop=True)

benign_trigrams = Counter()
total_benign_trigrams = 0
for sld in df_train.loc[df_train["label"] == 0, "sld"]:
    padded = f"^{sld}$"
    for i in range(len(padded) - 2):
        benign_trigrams[padded[i:i+3]] += 1
        total_benign_trigrams += 1

vocab_size = max(len(benign_trigrams), 1)

def benign_trigram_log_prob(s: str) -> float:
    padded = f"^{s}$"
    k = max(len(padded) - 2, 1)
    log_sum = 0.0
    for i in range(len(padded) - 2):
        tg = padded[i:i+3]
        prob = (benign_trigrams.get(tg, 0) + 1.0) / (total_benign_trigrams + vocab_size)
        log_sum += math.log2(prob)
    return log_sum / k

df_train["trigram_ll"] = df_train["sld"].apply(benign_trigram_log_prob)
df_test["trigram_ll"] = df_test["sld"].apply(benign_trigram_log_prob)

ALL_LEX_COLS = LEXICAL_COLS + ["trigram_ll"]

y_train = df_train["label"].values
y_test = df_test["label"].values

X_train_lex = df_train[ALL_LEX_COLS].values
X_test_lex = df_test[ALL_LEX_COLS].values

vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 4), max_features=300, sublinear_tf=True)
X_train_ng = vectorizer.fit_transform(df_train["sld"]).toarray()
X_test_ng = vectorizer.transform(df_test["sld"]).toarray()

X_train_hybrid = np.hstack([X_train_lex, X_train_ng])
X_test_hybrid = np.hstack([X_test_lex, X_test_ng])

ngram_names = [f"ng_{g}" for g in vectorizer.get_feature_names_out()]
HYBRID_FEATURE_NAMES = ALL_LEX_COLS + ngram_names

print(f"Train samples: {X_train_hybrid.shape[0]:,} | Test samples: {X_test_hybrid.shape[0]:,}")
print(f"Dense lexical shape: {X_train_lex.shape} | Dense hybrid shape: {X_train_hybrid.shape}")

Train samples: 16,000 | Test samples: 4,000
Dense lexical shape: (16000, 7) | Dense hybrid shape: (16000, 307)


### Model training and Hyperparameter tuning and evaluation

Comparison between 5 different classification configurations using the lexical features and hybrid lexical TF-IDF features, measuring the accuracy, precision, recall, F1-score, ROC-AUC, false-positive rate (FPR), and inference latency per 1,000 samples as evaluation metrics.

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)

param_dist = {
    "max_depth": [6, 8, 10],
    "learning_rate": [0.05, 0.1, 0.15],
    "n_estimators": [150, 250],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "subsample": [0.8, 1.0]
}

search_lex = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss"),
    param_distributions=param_dist, n_iter=5, scoring="f1", cv=3, random_state=42, n_jobs=-1
).fit(X_train_lex, y_train)

search_hyb = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss"),
    param_distributions=param_dist, n_iter=5, scoring="f1", cv=3, random_state=42, n_jobs=-1
).fit(X_train_hybrid, y_train)

print(f"Best params (7 Dense Feats): {search_lex.best_params_}")
print(f"Best params (307 Hybrid Feats): {search_hyb.best_params_}\n")

experiments = [
    ("Logistic Regression (7 Dense Feats)", LogisticRegression(max_iter=500), X_train_lex, X_test_lex),
    ("Random Forest (7 Dense Feats)", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1), X_train_lex, X_test_lex),
    ("XGBoost (7 Dense Feats)", search_lex.best_estimator_, X_train_lex, X_test_lex),
    ("Random Forest (Hybrid: Dense + TF-IDF)", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1), X_train_hybrid, X_test_hybrid),
    ("XGBoost (Hybrid: Dense + TF-IDF)", search_hyb.best_estimator_, X_train_hybrid, X_test_hybrid),
]

results_rows = []
trained_models = {}

for name, model, X_tr, X_te in experiments:
    if not hasattr(model, "classes_"):
        model.fit(X_tr, y_train)

    t_inf = time.perf_counter()
    y_pred = model.predict(X_te)
    inf_ms_per_1k = ((time.perf_counter() - t_inf) * 1000) / (X_te.shape[0] / 1000.0)

    y_prob = model.predict_proba(X_te)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    results_rows.append({
        "Configuration": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1_Score": round(f1_score(y_test, y_pred), 4),
        "ROC_AUC": round(roc_auc_score(y_test, y_prob), 4),
        "FPR": round(fp / (fp + tn), 4),
        "Latency_ms_per_1k": round(inf_ms_per_1k, 3)
    })
    trained_models[name] = (model, y_pred)

results_df = pd.DataFrame(results_rows)
print(results_df.to_string(index=False))

xgb_lex_model, xgb_lex_pred = trained_models["XGBoost (7 Dense Feats)"]
xgb_hyb_model, xgb_hyb_pred = trained_models["XGBoost (Hybrid: Dense + TF-IDF)"]

print("\nConfusion Matrix - XGBoost Hybrid:")
print(confusion_matrix(y_test, xgb_hyb_pred))

Best params (7 Dense Feats): {'subsample': 0.8, 'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
Best params (307 Hybrid Feats): {'subsample': 0.8, 'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.05, 'colsample_bytree': 0.6}

                         Configuration  Accuracy  Precision  Recall  F1_Score  ROC_AUC    FPR  Latency_ms_per_1k
   Logistic Regression (7 Dense Feats)    0.8452     0.8557  0.8305    0.8429   0.9133 0.1400              0.123
         Random Forest (7 Dense Feats)    0.8568     0.8635  0.8475    0.8554   0.9264 0.1340             30.318
               XGBoost (7 Dense Feats)    0.8632     0.8900  0.8290    0.8584   0.9356 0.1025              4.187
Random Forest (Hybrid: Dense + TF-IDF)    0.8775     0.9085  0.8395    0.8727   0.9364 0.0845             48.105
      XGBoost (Hybrid: Dense + TF-IDF)    0.8760     0.9017  0.8440    0.8719   0.9423 0.0920              9.584

Confusion Matrix - XGBoost Hybrid:
[[1816  184]
 [